# 03 - Additional Dataset Experiments

This notebook is reserved for experiments on datasets that are outside the paper scope.

Use this notebook for extension runs only. Official reproduction tables should stay in notebook 01 and notebook 02.


## Setup

The notebook stores extension results separately from the paper tables.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.scan_local import run_dataset_experiment

RESULT_DIR = ROOT / 'data' / 'scan_results' / 'additional_datasets'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR


## Experiment Config

Update the dataset name only after that dataset has been integrated into `src/scan_datasets.py`.


In [ ]:
DATASET_NAME = 'custom-dataset'
K0 = 10
ENABLE_PNP = True
RUN_EXPERIMENT = False

EPOCHS = 80
WARMUP_EPOCHS = 30
LAMBDA_PARAM = 2.0
BATCH_SIZE = 2048
LR = 5e-3
ENTROPY_WEIGHT = 5.0
INIT_STRATEGY = 'kmeans'

RESULT_CSV = RESULT_DIR / 'additional_dataset_runs.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Optional Run

This cell is intentionally guarded by `RUN_EXPERIMENT` so the notebook does not launch an unsupported run by accident.


In [ ]:
if RUN_EXPERIMENT:
    result = run_dataset_experiment(
        dataset_name=DATASET_NAME,
        k0=K0,
        enable_pnp=ENABLE_PNP,
        method_name='additional-dataset',
        device=DEVICE,
        epochs=EPOCHS,
        warmup_epochs=WARMUP_EPOCHS,
        lambda_param=LAMBDA_PARAM,
        batch_size=BATCH_SIZE,
        lr=LR,
        entropy_weight=ENTROPY_WEIGHT,
        init_strategy=INIT_STRATEGY,
    )

    row = pd.DataFrame([
        {
            'Dataset': DATASET_NAME,
            'K0': result.k0,
            'K*': result.inferred_k,
            'ACC(%)': result.acc * 100.0,
            'NMI(%)': result.nmi * 100.0,
            'ARI(%)': result.ari * 100.0,
        }
    ])

    if RESULT_CSV.exists():
        history = pd.read_csv(RESULT_CSV)
        history = pd.concat([history, row], ignore_index=True)
    else:
        history = row

    history.to_csv(RESULT_CSV, index=False)
    display(row)
else:
    print('Set RUN_EXPERIMENT = True after choosing a dataset that is already supported by src/scan_datasets.py.')


## Recorded Runs


In [ ]:
if RESULT_CSV.exists():
    history = pd.read_csv(RESULT_CSV)
    display(history)
else:
    print('No additional dataset runs recorded yet.')
